# Experiment 2: Bag of Words vs TF-IDF Comparison

This experiment evaluates multiple classification algorithms (Logistic Regression, MultinomialNB, XGBoost, Random Forest, Gradient Boosting) using both CountVectorizer (Bag of Words) and TfidfVectorizer on the balanced sentiment dataset.

In [ ]:
import os
import re
import string
import pandas as pd
import numpy as np
import scipy.sparse
import warnings

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Configuration & Experiment Setup

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

CONFIG = {
    "data_path": DATA_PATH,
    "test_size": 0.2,
    "mlflow_tracking_uri": os.getenv("MLFLOW_TRACKING_URI", "https://dagshub.com/rohitkr8527/sentiment-analysis.mlflow"),
    "dagshub_repo_owner": os.getenv("DAGSHUB_REPO_OWNER", "rohitkr8527"),
    "dagshub_repo_name": os.getenv("DAGSHUB_REPO_NAME", "sentiment-analysis"),
    "experiment_name": "Bow vs TfIdf"
}

# Setup MLflow if credentials are available
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
    if os.getenv("DAGSHUB_TOKEN"):
        dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
    mlflow.set_experiment(CONFIG["experiment_name"])
except Exception as e:
    print(f"MLflow setup skipped: {e}")

## 2. Text Preprocessing Utilities

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = ''.join([char for char in text if not char.isdigit()])
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', '')
    text = re.sub(r'\s+', ' ', text).strip()
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(words)

def load_data(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)
    df['text'] = df['text'].apply(preprocess_text)
    return df

## 3. Define Vectorizers and Classifiers

In [ ]:
VECTORIZERS = {
    'BoW': CountVectorizer(max_features=5000),
    'TF-IDF': TfidfVectorizer(max_features=5000)
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(eval_metric='logloss'),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

## 4. Run Experiments and Model Comparison

In [ ]:
df = load_data(CONFIG["data_path"])
results = []

for vec_name, vectorizer in VECTORIZERS.items():
    print(f"\n--- Feature Extraction: {vec_name} ---")
    X = vectorizer.fit_transform(df['text'])
    y = df['sentiment']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=42)

    for algo_name, algorithm in ALGORITHMS.items():
        print(f"Training {algo_name} with {vec_name}...")
        model = algorithm
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        results.append({
            "Algorithm": algo_name,
            "Vectorizer": vec_name,
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1-Score": round(f1, 4)
        })

results_df = pd.DataFrame(results)
print("\n--- Final Experiment Results ---")
display(results_df) if 'display' in globals() else print(results_df)